# PosCor — sample dataset (format demo only)

**This is not data from the PosCor study.** It is simulated data that follows the format the paper describes
(dual MPU-6050 IMUs at 50 Hz, 7 serpentine strain sensors, 6 posture classes, Vicon-style reference angles).
Use it to build and test your pipeline. Do not report results from it or present it as study data.

Sections: 1) generate the sample · 2) inspect · 3) window it (2 s, 50% overlap) · 4) quick sanity plots

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
FS = 50                      # Hz, as in the paper
SEED = 42
SUBJECTS = ["S01", "S02", "S03"]
SECONDS_PER_POSTURE = 11     # gives 10 windows of 2 s at 50% overlap
# class: (sagittal flexion deg, lateral bend deg [+left], sitting flag)
CLASSES = {"Sit_Slouch":(28,0,1), "Sit_BendLeft":(8,18,1), "Sit_BendRight":(8,-18,1),
           "Sit_Straight":(3,0,1), "Stand_Upright":(0,0,0), "Stand_Slouch":(22,0,0)}
R0 = np.array([1180,1210,1195,1220,1205,1190,1215.0])   # baseline resistances (ohm)

## 1. Generate the sample

In [ ]:
rng = np.random.default_rng(SEED)
def lowpass_noise(n, sd, k=15):
    x = rng.normal(0, sd, n + k)
    return np.convolve(x, np.ones(k)/k, mode="valid")[:n] * np.sqrt(k)

parts, t0 = [], 0.0
for sub in SUBJECTS:
    for label, (flex, bend, sit) in CLASSES.items():
        n = FS * SECONDS_PER_POSTURE
        t = np.arange(n) / FS
        # smooth "true" angles (sway + slow drift + per-trial offset)
        fl = flex + 1.5*np.sin(2*np.pi*0.25*t + rng.uniform(0, 6)) + lowpass_noise(n, 0.4) + rng.normal(0, 2)
        lb = bend + 0.8*np.sin(2*np.pi*0.20*t + rng.uniform(0, 6)) + lowpass_noise(n, 0.3) + rng.normal(0, 1.5)
        df = pd.DataFrame({"subject_id": sub, "posture_label": label, "time_s": np.round(t0 + t, 3)})
        for imu, gain in (("imu1", 1.0), ("imu2", 0.6)):          # upper back / lumbar
            f, b = np.radians(fl*gain), np.radians(lb*gain)
            g = 9.81
            df[f"{imu}_acc_x"] = g*np.sin(f) + rng.normal(0, .05, n)              # m/s^2
            df[f"{imu}_acc_y"] = g*np.sin(b) + rng.normal(0, .05, n)
            df[f"{imu}_acc_z"] = g*np.cos(f)*np.cos(b) + rng.normal(0, .05, n)
            df[f"{imu}_gyr_x"] = np.degrees(np.gradient(f)*FS) + rng.normal(0, .3, n)  # deg/s
            df[f"{imu}_gyr_y"] = np.degrees(np.gradient(b)*FS) + rng.normal(0, .3, n)
            df[f"{imu}_gyr_z"] = rng.normal(0, .3, n)
        # strain array, mirroring the paper's mappings:
        # theta_flex ~ rho4+rho5+rho6 ; theta_bend ~ (rho2+rho3)-(rho6+rho7)
        rho = np.zeros((n, 7))
        rho[:, 3:6] += (fl * 0.0013)[:, None]
        rho[:, 1:3] += np.clip(lb, 0, None)[:, None] * 0.003
        rho[:, 5:7] += np.clip(-lb, 0, None)[:, None] * 0.003
        rho[:, 4:7] += 0.02 * sit                       # seated lumbar loading
        rho[:, 0:3] += (fl * 0.0007 * (1 - sit))[:, None]
        rho += rng.normal(0, 0.002, (n, 7))
        R = R0 * (1 + rho) * (1 + rng.normal(0, 0.01))   # small per-trial fit offset
        for i in range(7):
            df[f"strain_R{i+1}_ohm"] = R[:, i]
        df["ref_flexion_deg"] = fl            # Vicon-style reference, resampled to 50 Hz
        df["ref_lateral_bend_deg"] = lb
        parts.append(df); t0 += t[-1] + 1/FS

data = pd.concat(parts, ignore_index=True).round(4)
data.insert(0, "SYNTHETIC", True)
data.to_csv("poscor_SYNTHETIC_sample.csv", index=False)
print(data.shape)

## 2. Inspect

In [ ]:
data.head()

In [ ]:
data.groupby(["subject_id","posture_label"]).size().unstack()

## 3. Window it (2 s windows, 50% overlap)
Arrays shaped `(windows, time, channels)`, ready for a CNN–BiLSTM.

In [ ]:
WIN, STEP = 2*FS, FS
IMU_COLS = [c for c in data.columns if c.startswith("imu")]
STRAIN_COLS = [c for c in data.columns if c.startswith("strain")]
Xi, Xs, y, groups = [], [], [], []
for (sub, lab), g in data.groupby(["subject_id","posture_label"], sort=False):
    for s in range(0, len(g) - WIN + 1, STEP):
        w = g.iloc[s:s+WIN]
        Xi.append(w[IMU_COLS].values); Xs.append(w[STRAIN_COLS].values)
        y.append(lab); groups.append(sub)
X_imu, X_strain = np.stack(Xi), np.stack(Xs)
labels = np.array(y); groups = np.array(groups)
print("IMU:", X_imu.shape, " Strain:", X_strain.shape, " windows per class:",
      pd.Series(labels).value_counts().to_dict())

## 4. Sanity plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
s1 = data[data.subject_id == "S01"]
axes[0].plot(s1.time_s, s1.ref_flexion_deg, label="flexion")
axes[0].plot(s1.time_s, s1.ref_lateral_bend_deg, label="lateral bend")
axes[0].set(xlabel="time (s)", ylabel="deg", title="S01 reference angles"); axes[0].legend()
rho = (data[STRAIN_COLS].values - R0) / R0
pd.DataFrame(rho, columns=[f"ρ{i}" for i in range(1, 8)]).assign(cls=data.posture_label)\
  .groupby("cls").mean().T.plot(ax=axes[1], marker="o")
axes[1].set(title="Mean normalised resistance change per sensor", xlabel="sensor")
plt.tight_layout(); plt.show()